In [ ]:
import cv2
from ultralytics import YOLO

# Load model YOLOv8
model = YOLO("yolov8n.pt")

# Buka video
video_path = "C:\\Pemrograman\\kuliah\\smt5\\computer_vision\\video_analytics\\video_input\\video2.mp4"
cap = cv2.VideoCapture(video_path)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

# Output video
out = cv2.VideoWriter(
    "output_people_count.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Deteksi orang (class 0 = person)
    results = model(frame, conf=0.4, classes=[0])

    person_count = 0

    for r in results:
        for box in r.boxes:
            person_count += 1
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    # Tampilkan jumlah orang
    cv2.putText(
        frame,
        f"Jumlah Orang: {person_count}",
        (40,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        2
    )

    out.write(frame)

cap.release()
out.release()
print("Selesai! Video tersimpan sebagai output_people_count.mp4")


0: 384x640 3 persons, 162.3ms
Speed: 4.7ms preprocess, 162.3ms inference, 3.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 108.9ms
Speed: 4.3ms preprocess, 108.9ms inference, 2.6ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 105.8ms
Speed: 5.9ms preprocess, 105.8ms inference, 2.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 182.3ms
Speed: 4.9ms preprocess, 182.3ms inference, 3.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 133.1ms
Speed: 4.5ms preprocess, 133.1ms inference, 3.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 110.4ms
Speed: 4.5ms preprocess, 110.4ms inference, 3.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 109.7ms
Speed: 4.7ms preprocess, 109.7ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 4 persons, 137.9ms
Speed: 4.8ms preprocess, 137.9ms inference, 4.0ms postprocess per 

In [10]:
import cv2
from ultralytics import YOLO

# Load model YOLOv8 pose untuk deteksi pose
model = YOLO("yolov8n-pose.pt")  # Pastikan model pose tersedia, jika tidak download dari ultralytics

# Buka video
video_path = "C:\\Pemrograman\\kuliah\\smt5\\computer_vision\\video_analytics\\video_input\\video3.mp4"
cap = cv2.VideoCapture(video_path)

width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps    = cap.get(cv2.CAP_PROP_FPS)

# Output video
out = cv2.VideoWriter(
    "output_people_raised_hands.mp4",
    cv2.VideoWriter_fourcc(*'mp4v'),
    fps,
    (width, height)
)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Deteksi pose orang
    results = model(frame, conf=0.4)

    person_count = 0
    raised_hands_count = 0

    for r in results:
        if r.keypoints is not None and len(r.keypoints.xy) > 0:
            num_persons = len(r.keypoints.xy)
            for i in range(num_persons):
                person_count += 1
                kp = r.keypoints.xy[i]  # shape (17, 2) untuk x,y
                conf = r.keypoints.conf[i]  # shape (17,) untuk confidence
                
                # Keypoints: 5=left_shoulder, 6=right_shoulder, 9=left_wrist, 10=right_wrist
                left_shoulder = kp[5]
                right_shoulder = kp[6]
                left_wrist = kp[9]
                right_wrist = kp[10]
                
                # Logika deteksi: wrist di atas shoulder dengan margin 20px, confidence > 0.6
                margin = 20  # pixel margin, dikurangi dari 50
                left_raised = (conf[9] > 0.6 and conf[5] > 0.6 and 
                               left_wrist[1] < left_shoulder[1] - margin)
                right_raised = (conf[10] > 0.6 and conf[6] > 0.6 and 
                                right_wrist[1] < right_shoulder[1] - margin)
                
                if left_raised or right_raised:
                    raised_hands_count += 1
                
                # Gambar bounding box jika ada
                if r.boxes is not None and i < len(r.boxes):
                    box = r.boxes[i]
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)

    # Tampilkan jumlah orang dan orang yang mengangkat tangan
    cv2.putText(
        frame,
        f"Jumlah Orang: {person_count}",
        (40,80),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (0,0,255),
        2
    )
    cv2.putText(
        frame,
        f"Orang Mengangkat Tangan: {raised_hands_count}",
        (40,120),
        cv2.FONT_HERSHEY_SIMPLEX,
        1,
        (255,0,0),
        2
    )

    out.write(frame)

cap.release()
out.release()
print("Selesai! Video tersimpan sebagai output_people_raised_hands.mp4")


0: 384x640 6 persons, 169.0ms
Speed: 32.6ms preprocess, 169.0ms inference, 20.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 111.0ms
Speed: 3.2ms preprocess, 111.0ms inference, 2.8ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 6 persons, 140.2ms
Speed: 3.6ms preprocess, 140.2ms inference, 2.9ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 96.6ms
Speed: 3.3ms preprocess, 96.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 99.2ms
Speed: 3.1ms preprocess, 99.2ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 99.7ms
Speed: 3.2ms preprocess, 99.7ms inference, 3.0ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 94.1ms
Speed: 4.0ms preprocess, 94.1ms inference, 2.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 5 persons, 95.5ms
Speed: 3.2ms preprocess, 95.5ms inference, 1.9ms postprocess per image at